# Проект спринта 7: Секреты Темнолесья

Команда онлайн-игры «Секретов Темнолесья» хочет написать статью о развитии индустрии игр в начале XXI века. Выпустить статью планируют на известном IT-ресурсе. По идее команды, статья должна привлечь внимание людей, которые любят старые игры, и заодно заинтересовать их «Секретами Темнолесья». 

## Цель проекта
Нужно изучить развитие игровой индустрии на основе исторических данных, которые команда собрала из открытых источников. В них есть информация о продажах игр, сделанных в разных жанрах и выпущенных на разных платформах, а также пользовательские и экспертные оценки игр.

**Задачи:**
1. Познакомиться с данными, проверить их корректность и провести предобработку, получив необходимый срез данных.
2. Отобрать данные по времени выхода игры (нам нужен период с 2000 по 2013 год включительно).
3. Категоризовать игры по оценкам пользователей и экспертов.
4. Выделить топ-7 платформ по количеству игр, выпущенных за весь требуемый период.

## Описание данных

Данные `/datasets/new_games.csv` содержат информацию о продажах игр разных жанров и платформ, а также пользовательские и экспертные оценки игр:
- `Name` — название игры.
- `Platform` — название платформы.
- `Year of Release` — год выпуска игры.
- `Genre` — жанр игры.
- `NA sales` — продажи в Северной Америке (в миллионах проданных копий).
- `EU sales` — продажи в Европе (в миллионах проданных копий).
- `JP sales` — продажи в Японии (в миллионах проданных копий).
- `Other sales` — продажи в других странах (в миллионах проданных копий).
- `Critic Score` — оценка критиков (от 0 до 100).
- `User Score` — оценка пользователей (от 0 до 10).
- `Rating` — рейтинг организации ESRB (англ. Entertainment Software Rating Board). Эта ассоциация определяет рейтинг компьютерных игр и присваивает им подходящую возрастную категорию.

## Содержание
1. [Загрузка и знакомство с данными](#1-bullet)
2. [Проверка ошибок в данных и их предобработка](#2-bullet)
   - [Обрабатываем пропуски в данных](#2.1-bullet)
   - [Корректируем типы данных в столбцах](#2.2-bullet)
   - [Работаем с дубликатами](#2.3-bullet)
3. [Фильтрация данных](#3-bullet)
4. [Категоризация данных](#4-bullet)
   - [Выделяем категории в данных](#4.1-bullet)
   - [Составляем топ-7 платформ по количеству игр](#4.2-bullet)
5. [Общий вывод из исследования](#5-bullet)

<a class="anchor" id="1-bullet"></a>
## 1. Загрузка и знакомство с данными

Загрузим необходимые библиотеки Python и данные датасета `/datasets/new_games.csv`.

In [1]:
import pandas as pd

In [2]:
data = pd.read_csv('https://code.s3.yandex.net/datasets/new_games.csv')
games = pd.DataFrame(data)

Познакомимся с данными: выведим первые строки и результат метода `info()`.

In [3]:
games.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16954 non-null  object 
 1   Platform         16956 non-null  object 
 2   Year of Release  16681 non-null  float64
 3   Genre            16954 non-null  object 
 4   NA sales         16956 non-null  float64
 5   EU sales         16956 non-null  object 
 6   JP sales         16956 non-null  object 
 7   Other sales      16956 non-null  float64
 8   Critic Score     8242 non-null   float64
 9   User Score       10152 non-null  object 
 10  Rating           10085 non-null  object 
dtypes: float64(4), object(7)
memory usage: 1.4+ MB


In [32]:
print(games.head())

,name,platform,year_of_release,genre,sales_north_america,sales_europe,sales_japan,sales_other,critic_score,user_score,rating
0,Wii Sports,Wii,2006,sports,41.36,28.96,3.77,8.45,76,8.0,76.0
1,Super Mario Bros.,NES,1985,platform,29.08,3.58,6.81,0.77,-1,-1.0,-1.0
2,Mario Kart Wii,Wii,2008,racing,15.68,12.76,3.79,3.29,82,8.3,82.0
3,Wii Sports Resort,Wii,2009,sports,15.61,10.93,3.28,2.95,80,8.0,80.0
4,Pokemon Red/Pokemon Blue,GB,1996,role-playing,11.27,8.89,10.22,1.00,-1,-1.0,-1.0


**Выводы по данным.** 

В датасете 16956 записей и 11 столбцов. 

Данные соответствуют описанию, но примерно в половине столбцов есть пропуски. 

Названия столбцов отражают содержимое данных, однако для удобства их можно привести к формату snake case. 

Среди типов данных много некорректных. Например, в столбцах `EU sales`, `JP sales`, `User Score` и `Rating` тип данных `object`, хотя сами значения там числовые. Вероятно, такой тип данных присвоился столбцам из-за того, что в них встречаются пропуски или строковые значения. Также в столбце `Year of Release` неоптимальный тип данных.

Типы данных можно было бы исправить так:
- В столбцах `EU sales`, `JP sales`, `User Score` обработать пропуски или строковые значения, а затем привести их к типу данных `float64`.
- В столбцах `Platform`, `Rating` и `Genre` количество возможных значений рейтинга небольшое, поэтому их можно привести к типу `category`.
- Столбец `Year of Release` можно привести к типу `int16`, `datetime64` или `category` в зависимости от того, какие операции может понадобиться совершить с этими значениями.
- Столбец `Critic Score` стоит проверить на наличие дробных чисел: если они есть, то оставить в столбце текущий тип `float64`, а если нет, то изменить на `int8`.

<a class="anchor" id="2-bullet"></a>
## 2. Проверка ошибок в данных и их предобработка

Выведем названия всех столбцов.

In [5]:
print(games.columns)

Index(['Name', 'Platform', 'Year of Release', 'Genre', 'NA sales', 'EU sales',
       'JP sales', 'Other sales', 'Critic Score', 'User Score', 'Rating'],
      dtype='object')


Названия столбцов не соответствуют формату snake case. Приведём их к нему и совсем немного изменим для удобства.

In [6]:
columns_snake_case = ['name', 'platform', 'year_of_release', 'genre', 'sales_north_america', 'sales_europe',
       'sales_japan', 'sales_other', 'critic_score', 'user_score', 'rating']

games.columns = columns_snake_case

print(games.columns)

Index(['name', 'platform', 'year_of_release', 'genre', 'sales_north_america',
       'sales_europe', 'sales_japan', 'sales_other', 'critic_score',
       'user_score', 'rating'],
      dtype='object')


Так как на предыдущем этапе мы нашли много некорректных типов данных, их нужно изменить. Но с некоторыми столбцами это невозможно сделать на этом этапе, пока в них есть пропуски, поэтому сначала поработаем с пропусками.

<a class="anchor" id="2.1-bullet"></a>
### Обрабатываем пропуски в данных

Выясним, сколько в данных пропусков в абсолютном и относительном значениях.

In [7]:
# На будущее нам понадобится количество строк в датафрейме, вычислим его здесь и присвоим переменной, чтобы дальше переиспользовать
all_data = games.shape[0]

# Выведем пропуски в данных в абсолютном и относительном значении
print(games.isna().sum(), games.isna().sum() / len(games) * 100, sep='\n\n')

name                      2
platform                  0
year_of_release         275
genre                     2
sales_north_america       0
sales_europe              0
sales_japan               0
sales_other               0
critic_score           8714
user_score             6804
rating                 6871
dtype: int64

name                    0.011795
platform                0.000000
year_of_release         1.621845
genre                   0.011795
sales_north_america     0.000000
sales_europe            0.000000
sales_japan             0.000000
sales_other             0.000000
critic_score           51.391838
user_score             40.127389
rating                 40.522529
dtype: float64


**Выводы по пропускам.** Больше всего пропусков в столбцах с *оценками от критиков и зрителей*, а также с *рейтингом игры* (7-8 тыс. пропусков, или 40-50% от всех значений). Причины могут быть такими:
1. Так как данные собраны из разных источников, у игр нет унифицированных названий. В один датасет могли попасть названия одной и той же игры, которые отличаются заглавной буквой, точкой, опечаткой, переводом, версией для разных платформ. Часть строк для игр-дубликатов могла не совместиться, и поля остались пустыми. Это можно проверить, посмотрев количество явных и неявных дубликатов в данных.
2. У старых игр могло не быть единой системы оценок, а рейтинг не фиксировался в открытых источниках, поэтому нормальных данных о них просто нет. Можно посмотреть распределение пропусков по годам: если они в основном в 2000-2005 годах, значит причина может быть в этом.
3. Возможно, игры с пропусками были малоизвестны и действительно не получили ни оценок, ни рейтинга. Эту гипотезу можно подтвердить или опровергнуть, посмотрев на столбцы со значениями продаж.
4. Возможно, не все игры получают ESRB рейтинг, а также не все существующие сейчас категории рейтинга были созданы к 2000-му году (например, E10+ принят в 2005-м).
5. Если команда собирала данные вручную, то возможны ошибки, связанные с невнимательностью, а также ошибки загрузки.

**Попробуем поработать с пропусками.**

В столбце `name` всего 2 пропуска из всего датафрейма (0.01% от всех данных). Можем смело удалить их, тем более что без названий мы никак не сможем идентифицировать игры и вряд ли они будут полезны в дальнейшем анализе.

В столбце `year_of_release` 275 пропусков. Это всего 1.6% от всех данных, но стоит попробовать их заполнить. Например, если год пропущен у игры для одной платформы, возможно, получится заполнить его на основе данных этой же игры для других платформ. Оставшиеся данные придётся удалить, так как в дальнейшем мы не сможем их отфильтровать по нужному временному периоду.

В столбце `genre` 2 пропуска (0.01% от всех данных). Попробуем заполнить их так же на основе данных той же игры для других платформ, а если не получится, удалим.

В столбцах `critic_score`, `user_score` и `rating` больше всего пропусков. Попытаемся восстановить их аналогично предыдущим пунктам - через ту же игру для других платформ. То, что не получится восстановить, заменим на -1, чтобы привести столбцы к числовому типу данных и совершать с данными операции как с числами.

**Итого:**
- `name` - удалить пропуски

In [8]:
print(f'До удаления пропусков в name: ', games.isna().sum(), sep='\n', end='\n\n')
games = games[~games['name'].isna()]
print(f'После удаления пропусков в name: ', games.isna().sum(), sep='\n', end='\n\n')

До удаления пропусков в name: 
name                      2
platform                  0
year_of_release         275
genre                     2
sales_north_america       0
sales_europe              0
sales_japan               0
sales_other               0
critic_score           8714
user_score             6804
rating                 6871
dtype: int64

После удаления пропусков в name: 
name                      0
platform                  0
year_of_release         275
genre                     0
sales_north_america       0
sales_europe              0
sales_japan               0
sales_other               0
critic_score           8712
user_score             6802
rating                 6869
dtype: int64



- `year_of_release` - попробовать заполнить, остальное удалить (пропуски в `genre` удалились сами вместе с пропусками в `name`)

In [9]:
# Создаём функцию для заполнения пропущенных годов выпуска на основе года выпуска игр с таким же названием
def fill_year_by_name(row):
    if pd.isna(row['year_of_release']):
        value = games[games['name'] == row['name']]['year_of_release']
        if not value.empty:
            return value.iloc[0]
        else:
            return pd.NA
    else:
        return row['year_of_release']

# Применяем функцию ко всем строкам датасета
games['year_of_release'] = games.apply(fill_year_by_name, axis=1)

print(f'После заполнения пропусков в year_of_release: ', games.isna().sum(), sep='\n', end='\n\n') 

# Удаляем то, что не получилось заполнить
games = games[~games['year_of_release'].isna()]

print(f'После удаления пропусков в year_of_release: ', games.isna().sum(), sep='\n', end='\n\n') 

После заполнения пропусков в year_of_release: 
name                      0
platform                  0
year_of_release         195
genre                     0
sales_north_america       0
sales_europe              0
sales_japan               0
sales_other               0
critic_score           8712
user_score             6802
rating                 6869
dtype: int64

После удаления пропусков в year_of_release: 
name                      0
platform                  0
year_of_release           0
genre                     0
sales_north_america       0
sales_europe              0
sales_japan               0
sales_other               0
critic_score           8621
user_score             6730
rating                 6799
dtype: int64



- `critic_score`, `user_score`, `rating` - попробовать заполнить, остальное заменить на -1

In [10]:
# Создаём функцию для заполнения пропущенных оценок критиков на основе названия и года выпуска игр
def fill_critic_score_by_name(row):
    if pd.isna(row['critic_score']):
        value = games[(games['name'] == row['name']) & 
            games['year_of_release'] == row['year_of_release']]['critic_score']
        if not value.empty:
            return value.iloc[0]
        else:
            return pd.NA
    else:
        return row['critic_score']

# Применяем функцию ко всем строкам датасета
games['critic_score'] = games.apply(fill_critic_score_by_name, axis=1)

print(f'После заполнения пропусков в critic_score: ', games.isna().sum(), sep='\n', end='\n\n') 

# Остальное заменяем на -1
games['critic_score'] = games['critic_score'].fillna(-1)

print(f'После удаления пропусков в critic_score: ', games.isna().sum(), sep='\n', end='\n\n') 

После заполнения пропусков в critic_score: 
name                      0
platform                  0
year_of_release           0
genre                     0
sales_north_america       0
sales_europe              0
sales_japan               0
sales_other               0
critic_score           8621
user_score             6730
rating                 6799
dtype: int64

После удаления пропусков в critic_score: 
name                      0
platform                  0
year_of_release           0
genre                     0
sales_north_america       0
sales_europe              0
sales_japan               0
sales_other               0
critic_score              0
user_score             6730
rating                 6799
dtype: int64



/var/folders/wx/gd5mkv752w9558dws3m728_00000gn/T/ipykernel_32044/2984391590.py:19: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  games['critic_score'] = games['critic_score'].fillna(-1)


In [11]:
# Создаём функцию для заполнения пропущенных оценок пользователей на основе названия игры
def fill_user_score_by_name(row):
    if pd.isna(row['user_score']):
        value = games[games['name'] == row['name']]['user_score']
        if not value.empty:
            return value.iloc[0]
        else:
            return pd.NA
    else:
        return row['user_score']

# Применяем функцию ко всем строкам датасета
games['user_score'] = games.apply(fill_user_score_by_name, axis=1)

print(f'После заполнения пропусков в user_score: ', games.isna().sum(), sep='\n', end='\n\n') 

# Остальное заменяем на -1
games['user_score'] = games['user_score'].fillna(-1)

print(f'После удаления пропусков в user_score: ', games.isna().sum(), sep='\n', end='\n\n') 

После заполнения пропусков в user_score: 
name                      0
platform                  0
year_of_release           0
genre                     0
sales_north_america       0
sales_europe              0
sales_japan               0
sales_other               0
critic_score              0
user_score             6465
rating                 6799
dtype: int64

После удаления пропусков в user_score: 
name                      0
platform                  0
year_of_release           0
genre                     0
sales_north_america       0
sales_europe              0
sales_japan               0
sales_other               0
critic_score              0
user_score                0
rating                 6799
dtype: int64



In [12]:
# Создаём функцию для заполнения пропущенного рейтинга ESRB на основе названия игры
def fill_rating_by_name(row):
    if pd.isna(row['rating']):
        value = games[games['name'] == row['name']]['rating']
        if not value.empty:
            return value.iloc[0]
        else:
            return pd.NA
    else:
        return row['rating']

# Применяем функцию ко всем строкам датасета
games['rating'] = games.apply(fill_critic_score_by_name, axis=1)

print(f'После заполнения пропусков в rating: ', games.isna().sum(), sep='\n', end='\n\n') 

# Остальное заменяем на -1
games['rating'] = games['rating'].fillna(-1)

print(f'После удаления пропусков в rating: ', games.isna().sum(), sep='\n', end='\n\n') 

После заполнения пропусков в rating: 
name                   0
platform               0
year_of_release        0
genre                  0
sales_north_america    0
sales_europe           0
sales_japan            0
sales_other            0
critic_score           0
user_score             0
rating                 0
dtype: int64

После удаления пропусков в rating: 
name                   0
platform               0
year_of_release        0
genre                  0
sales_north_america    0
sales_europe           0
sales_japan            0
sales_other            0
critic_score           0
user_score             0
rating                 0
dtype: int64



Теперь, когда пропусков в данных нет, можно привести столбцы к корректным типам данных.

<a class="anchor" id="2.2-bullet"></a>
### Корректируем типы данных в столбцах

Напомним, как можно исправить типы данных:
1. Столбцы `sales_europe`, `sales_japan`, `user_score` - к типу `float64`.
2. Столбцы `platform`, `rating` и `genre` - к типу `category`.
3. Столбец `year_of_release` - к типу `int16`, `datetime64` или `category` в зависимости от того, какие операции может понадобиться совершить с этими значениями.
4. Столбец `critic_score` - к типу `int8`, если в нём нет дробных чисел.

Итак, начнём работу. 
1. Приведём столбцы `sales_europe`, `sales_japan` и `user_score` к типу float64.

Начнём с `sales_europe`.

In [13]:
# Посмотрим на тип данных до преобразования
print('Тип данных sales_europe до преобразования: ', games['sales_europe'].dtype)

# Если попытаться выполнить код games['sales_europe'] = games['sales_europe'].astype('float64'), возникает ошибка: 
# в столбце есть строковое значение 'unknown', которое не даёт присвоить столбцу числовой тип. Это значение нужно заменить.

# Заменим сначала 'unknown' на -1, чтобы была возможность привести столбец к числовому типу данных
games['sales_europe'] = games['sales_europe'].replace('unknown', -1)

# Приведём столбец к нужному типу данных и убедимся, что всё получилось
games['sales_europe'] = games['sales_europe'].astype('float64')
print('Тип данных sales_europe после преобразования: ', games['sales_europe'].dtype)

# Для более точной аналитики заменим значение -1 на среднее значение в зависимости от названия платформы и года выхода игры
def replasement_europe(row):
    if row['sales_europe'] == -1:
        group = games[(row['platform'] == games['platform'])
            & (row['year_of_release'] == games['year_of_release'])]
        return group['sales_europe'].mean()
    else:
        return row['sales_europe']

# Применим функцию ко всем строкам
games['sales_europe'] = games.apply(replasement_europe, axis=1)

Тип данных sales_europe до преобразования:  object
Тип данных sales_europe после преобразования:  float64


Повторим то же самое с `sales_japan`.

In [14]:
# Посмотрим на тип данных до преобразования
print('Тип данных sales_japan до преобразования: ', games['sales_japan'].dtype)

# Если попытаться выполнить код games['sales_europe'] = games['sales_europe'].astype('float64'), возникает ошибка: 
# в столбце есть строковое значение 'unknown', которое не даёт присвоить столбцу числовой тип. Это значение нужно заменить.

# Заменим 'unknown' на -1
games['sales_japan'] = games['sales_japan'].replace('unknown', -1)

# Приведём столбец к нужному типу данных и убедимся, что всё получилось
games['sales_japan'] = games['sales_japan'].astype('float64')
print('Тип данных sales_japan после преобразования: ', games['sales_japan'].dtype)

# Заменим значение -1 на среднее значение в зависимости от названия платформы и года выхода игры
def replasement_europe(row):
    if row['sales_japan'] == -1:
        group = games[(row['platform'] == games['platform'])
            & (row['year_of_release'] == games['year_of_release'])]
        return group['sales_japan'].mean()
    else:
        return row['sales_japan']

# Применим функцию ко всем строкам
games['sales_japan'] = games.apply(replasement_europe, axis=1)

Тип данных sales_japan до преобразования:  object
Тип данных sales_japan после преобразования:  float64


Поработаем с `user_score`.

In [15]:
# Посмотрим на тип данных до преобразования
print('Тип данных user_score до преобразования: ', games['user_score'].dtype)

# При попытке привести столбец к типу float64 возникает ошибка: в столбце есть строковое значение 'tbd', 
# которое не даёт присвоить столбцу числовой тип. Это значение нужно заменить.

# Заменим 'tbd' на -1
games['user_score'] = games['user_score'].replace('tbd', -1)

# Приведём столбец к нужному типу данных и убедимся, что всё получилось
games['user_score'] = games['user_score'].astype('float64')
print('Тип данных user_score после преобразования: ', games['user_score'].dtype)

Тип данных user_score до преобразования:  object
Тип данных user_score после преобразования:  float64


2. Переходим к столбцам `platform`, `rating` и `genre`.

Начнём с `platform`.

In [16]:
# Посмотрим на тип данных до преобразования
print('Тип данных platform до преобразования: ', games['platform'].dtype)

# Выведем все уникальные значения столбца, чтобы убедиться, что их немного и будет уместно использовать тип данных category
print('Уникальные рейтинги ESRB: ', games['platform'].unique())

# Уникальных значений мало, присвоим столбцу тип данных category и убедимся, что всё получилось
games['platform'] = games['platform'].astype('category')
print('Тип данных platform после преобразования: ', games['platform'].dtype)

Тип данных platform до преобразования:  object
Уникальные рейтинги ESRB:  ['Wii' 'NES' 'GB' 'DS' 'X360' 'PS3' 'PS2' 'SNES' 'GBA' 'PS4' '3DS' 'N64'
 'PS' 'XB' 'PC' '2600' 'PSP' 'XOne' 'WiiU' 'GC' 'GEN' 'DC' 'PSV' 'SAT'
 'SCD' 'WS' 'NG' 'TG16' '3DO' 'GG' 'PCFX']
Тип данных platform после преобразования:  category


Повторим с `rating`.

In [17]:
# Посмотрим на тип данных до преобразования
print('Тип данных rating до преобразования: ', games['rating'].dtype)

# Выведем все уникальные значения столбца, чтобы убедиться, что их немного и будет уместно использовать тип данных category
print('Уникальные рейтинги ESRB: ', games['rating'].unique())

# Уникальных значений мало, присвоим столбцу тип данных category и убедимся, что всё получилось
games['rating'] = games['rating'].astype('category')
print('Тип данных rating после преобразования: ', games['rating'].dtype)


Тип данных rating до преобразования:  float64
Уникальные рейтинги ESRB:  [76. -1. 82. 80. 89. 58. 87. 91. 61. 97. 95. 77. 88. 83. 94. 93. 85. 86.
 98. 96. 90. 84. 73. 74. 78. 92. 71. 72. 68. 62. 49. 67. 81. 66. 56. 79.
 70. 59. 64. 75. 60. 63. 69. 50. 25. 42. 44. 55. 48. 57. 29. 47. 65. 54.
 20. 53. 37. 38. 33. 52. 30. 32. 43. 45. 51. 40. 46. 39. 34. 35. 41. 36.
 28. 31. 27. 26. 19. 23. 24. 21. 17. 13.]
Тип данных rating после преобразования:  category


Повторим с `genre`.

In [18]:
# Посмотрим на тип данных до преобразования
print('Тип данных genre до преобразования: ', games['genre'].dtype)

# Выведем все уникальные значения столбца, чтобы убедиться, что их немного и будет уместно использовать тип данных category
print('Уникальные жанры игр до преобразования: ', games['genre'].unique(), end='\n\n')

# Видно, что часть значений повторяется, но с ними мы разберёмся в разделе "Работаем с дубликатами" 

# Уникальных значений мало, присвоим столбцу тип данных category и убедимся, что всё получилось
games['genre'] = games['genre'].astype('category')
print('Тип данных genre после преобразования: ', games['genre'].dtype)
print('Уникальные жанры игр после преобразования: ', games['genre'].unique())

Тип данных genre до преобразования:  object
Уникальные жанры игр до преобразования:  ['Sports' 'Platform' 'Racing' 'Role-Playing' 'Puzzle' 'Misc' 'Shooter'
 'Simulation' 'Action' 'Fighting' 'Adventure' 'Strategy' 'MISC'
 'ROLE-PLAYING' 'RACING' 'ACTION' 'SHOOTER' 'FIGHTING' 'SPORTS' 'PLATFORM'
 'ADVENTURE' 'SIMULATION' 'PUZZLE' 'STRATEGY']

Тип данных genre после преобразования:  category
Уникальные жанры игр после преобразования:  ['Sports', 'Platform', 'Racing', 'Role-Playing', 'Puzzle', ..., 'PLATFORM', 'ADVENTURE', 'SIMULATION', 'PUZZLE', 'STRATEGY']
Length: 24
Categories (24, object): ['ACTION', 'ADVENTURE', 'Action', 'Adventure', ..., 'Shooter', 'Simulation', 'Sports', 'Strategy']


3. Переходим к `year_of_release`.

In [19]:
# Так как позднее мы будем отбирать временной период с 2000 по 2013 годы, будет удобнее привести столбец к формату int16 
# (datetime64 будет слишком громоздким, а category не позволит выбрать промежуток между годами)

# Посмотрим на тип данных до преобразования
print('Тип данных year_of_release до преобразования: ', games['year_of_release'].dtype)

# Проверим, за какие годы у нас вообще есть данные
print('Уникальные годы выхода игр до преобразования: ', games['year_of_release'].unique())

# Присвоим столбцу тип данных int16 и убедимся, что всё получилось
games['year_of_release'] = games['year_of_release'].astype('int16')
print('Тип данных year_of_release после преобразования: ', games['year_of_release'].dtype)
print('Уникальные годы выхода игр после преобразования: ', games['year_of_release'].unique())

Тип данных year_of_release до преобразования:  float64
Уникальные годы выхода игр до преобразования:  [2006. 1985. 2008. 2009. 1996. 1989. 1984. 2005. 1999. 2007. 2010. 2013.
 2004. 1990. 1988. 2002. 2001. 2011. 1998. 2015. 2012. 2014. 1992. 1997.
 1993. 1994. 1982. 2016. 2003. 1986. 2000. 1995. 1991. 1981. 1987. 1980.
 1983.]
Тип данных year_of_release после преобразования:  int16
Уникальные годы выхода игр после преобразования:  [2006 1985 2008 2009 1996 1989 1984 2005 1999 2007 2010 2013 2004 1990
 1988 2002 2001 2011 1998 2015 2012 2014 1992 1997 1993 1994 1982 2016
 2003 1986 2000 1995 1991 1981 1987 1980 1983]


4. И последнее изменение - столбец `critic_score`.

In [20]:
# Посмотрим на тип данных до преобразования
print('Тип данных critic_score до преобразования: ', games['critic_score'].dtype)

# Приведём столбец к нужному типу данных и убедимся, что всё получилось
games['critic_score'] = games['critic_score'].astype('int8')
print('Тип данных critic_score после преобразования: ', games['critic_score'].dtype)

Тип данных critic_score до преобразования:  float64
Тип данных critic_score после преобразования:  int8


<a class="anchor" id="2.3-bullet"></a>
### Работаем с дубликатами

Ещё раз посмотрим на категориальные данные: жанр игры, платформа, рейтинг и год выпуска. Нужно убедиться, что в них нет неявных дубликатов, связанных с опечатками или разным способом написания. Если такие есть - привести их к единообразию:

- `platform`

In [21]:
# Выведем все уникальные значения столбца platform
print('Уникальные платформы: ', games['platform'].unique(), end='\n\n')

# Отсортируем их в лексикографическом порядке
sorted_platforms = games['platform'].sort_values().unique()
print('Отсортированные уникальные платформы: ', sorted_platforms, end='\n\n')

# Дубликатов нет, только разные версии одних платформ, так что оставляем всё как есть

Уникальные платформы:  ['Wii', 'NES', 'GB', 'DS', 'X360', ..., 'NG', 'TG16', '3DO', 'GG', 'PCFX']
Length: 31
Categories (31, object): ['2600', '3DO', '3DS', 'DC', ..., 'WiiU', 'X360', 'XB', 'XOne']

Отсортированные уникальные платформы:  ['2600', '3DO', '3DS', 'DC', 'DS', ..., 'Wii', 'WiiU', 'X360', 'XB', 'XOne']
Length: 31
Categories (31, object): ['2600', '3DO', '3DS', 'DC', ..., 'WiiU', 'X360', 'XB', 'XOne']



- `rating`

In [22]:
# Выведем все уникальные значения столбца rating
print('Уникальные рейтинги ESRB: ', games['rating'].unique(), end='\n\n')

# В данных есть рейтинг 'K-A' - старый рейтинг, использовавшийся до 1998 года. Позже он был переименован в 'E', так что и мы заменим его
games['rating'] = games['rating'].replace('K-A', 'E')

# Также среди рейтингов есть значение -1. Посмотрим, сколько таких строк - если меньше 1%, его можно будет удалить
ratings = pd.Series(games['rating'])
print('Количество игр разных категорий: ', ratings.value_counts(), end='\n\n')

# Неизвестных значений очень много, а с рейтингом мы не будем работать в процессе анализа, поэтому оставим всё как есть

# Выведем данные после преобразования
print('Уникальные рейтинги ESRB после преобразования: ', games['rating'].unique())

Уникальные рейтинги ESRB:  [76.0, -1.0, 82.0, 80.0, 89.0, ..., 23.0, 24.0, 21.0, 17.0, 13.0]
Length: 82
Categories (82, float64): [-1.0, 13.0, 17.0, 19.0, ..., 95.0, 96.0, 97.0, 98.0]

Количество игр разных категорий:  rating
-1.0     8621
 70.0     254
 71.0     252
 75.0     245
 73.0     242
         ... 
 20.0       3
 29.0       3
 17.0       1
 21.0       1
 13.0       1
Name: count, Length: 82, dtype: int64

Уникальные рейтинги ESRB после преобразования:  [76.0, -1.0, 82.0, 80.0, 89.0, ..., 23.0, 24.0, 21.0, 17.0, 13.0]
Length: 82
Categories (82, float64): [-1.0, 13.0, 17.0, 19.0, ..., 95.0, 96.0, 97.0, 98.0]


- `genre`

In [23]:
# Выведем все уникальные значения столбца genre
print('Уникальные жанры игр: ', games['genre'].unique(), end='\n\n')

# Отсортируем их в лексикографическом порядке
sorted_genres = games['genre'].sort_values().unique()
print('Отсортированные уникальные платформы: ', sorted_genres, end='\n\n')

# Видно, что часть данных повторяется. Приведём все названия к нижнему регистру, чтобы получить именнно уникальные категории
def lowering(row):
    return row['genre'].lower()

games['genre'] = games.apply(lowering, axis=1)

# Выведем данные после преобразования
print('Уникальные жанры после преобразования: ', games['genre'].unique())

Уникальные жанры игр:  ['Sports', 'Platform', 'Racing', 'Role-Playing', 'Puzzle', ..., 'PLATFORM', 'ADVENTURE', 'SIMULATION', 'PUZZLE', 'STRATEGY']
Length: 24
Categories (24, object): ['ACTION', 'ADVENTURE', 'Action', 'Adventure', ..., 'Shooter', 'Simulation', 'Sports', 'Strategy']

Отсортированные уникальные платформы:  ['ACTION', 'ADVENTURE', 'Action', 'Adventure', 'FIGHTING', ..., 'STRATEGY', 'Shooter', 'Simulation', 'Sports', 'Strategy']
Length: 24
Categories (24, object): ['ACTION', 'ADVENTURE', 'Action', 'Adventure', ..., 'Shooter', 'Simulation', 'Sports', 'Strategy']

Уникальные жанры после преобразования:  ['sports' 'platform' 'racing' 'role-playing' 'puzzle' 'misc' 'shooter'
 'simulation' 'action' 'fighting' 'adventure' 'strategy']


- `year_of_release`

In [24]:
# Выведем все уникальные значения столбца year_of_release
print('Уникальные годы выпуска: ', games['year_of_release'].unique(), end='\n\n')

# Отсортируем их в лексикографическом порядке
sorted_years = games['year_of_release'].sort_values().unique()
print('Отсортированные годы выпуска: ', sorted_years)

# Дубликатов нет - оставляем всё как есть

Уникальные годы выпуска:  [2006 1985 2008 2009 1996 1989 1984 2005 1999 2007 2010 2013 2004 1990
 1988 2002 2001 2011 1998 2015 2012 2014 1992 1997 1993 1994 1982 2016
 2003 1986 2000 1995 1991 1981 1987 1980 1983]

Отсортированные годы выпуска:  [1980 1981 1982 1983 1984 1985 1986 1987 1988 1989 1990 1991 1992 1993
 1994 1995 1996 1997 1998 1999 2000 2001 2002 2003 2004 2005 2006 2007
 2008 2009 2010 2011 2012 2013 2014 2015 2016]


Теперь проверим наличие явных дубликатов во всех строках и удалим их.

In [25]:
# Отсортируем строки по названиям игр в лексикографическом порядке
sorted_games = games.sort_values(by='name')
print('Исходный отсортированный датафрейм: ', sorted_games[['name', 'platform', 'year_of_release']].head(20), sep='\n', end='\n\n')

# Посчитаем количество явных дубликатов
duplicated_rows = sorted_games.duplicated().sum()
all_rows = games.shape[0]
print(f'Общее количество строк: {all_rows}', f'Из них игр-дубликатов: {duplicated_rows}', sep='\n')

# Удалим явные дубликаты
games_no_duplicates = sorted_games.drop_duplicates()
all_rows_no_duplicates = games_no_duplicates.shape[0]
print(f'Количество строк после удаления дубликатов: {all_rows_no_duplicates}', end='\n\n')
print('Датафрейм после удаления дубликатов: ', games_no_duplicates[['name', 'platform', 'year_of_release']].head(20), sep='\n')

Исходный отсортированный датафрейм: 
                                              name platform  year_of_release
15192                               Beyblade Burst      3DS             2016
15191                               Beyblade Burst      3DS             2016
1086                             Fire Emblem Fates      3DS             2015
3906                          Frozen: Olaf's Quest       DS             2013
3394                          Frozen: Olaf's Quest      3DS             2013
13983                   Haikyu!! Cross Team Match!      3DS             2016
2478                             Tales of Xillia 2      PS3             2012
4782                                   '98 Koshien       PS             1998
8460                    .hack//G.U. Vol.1//Rebirth      PS2             2006
7182                  .hack//G.U. Vol.2//Reminisce      PS2             2006
8719       .hack//G.U. Vol.2//Reminisce (jp sales)      PS2             2006
8410                 .hack//G.U. Vol.3/

После того как мы удалили явные дубликаты, встаёт вопрос: нужны ли нам для анализа неполные дубликаты, в том числе версии одной игры для разных платформ? Например, в датафрейме выше видно, что игра `Frozen: Olaf's Quest` вышла на двух платформах: `DS` и `3DS`. В принципе, экспириенс пользователей на разных платформах мог отличаться, плюс нам нужно выделить топ-7 платформ по количеству выпущенных игр, так что "срезать" платформы не будем и оставим все данные.

Посчитаем количество строк, которые мы удалили за всё время обработки данных, в абсолютном и относительном значениях:

In [26]:
# Выведем исходное количество строк, которое мы считали в начале работы над проектом
print('Исходное количество строк в датасете: ', all_data)

# Выведем текущее количество строк
print('Количество строк после предобработки данных: ', games_no_duplicates.shape[0], end='\n\n')

# Посчитаем количество удалённых строк в абсолютном значении
deleted_count = all_data - games_no_duplicates.shape[0]
print('Количество удалённых строк: ', deleted_count)

# Посчитаем количество удалённых строк в относительном значении
deleted_share = round(deleted_count / all_data, 4)
print('Доля удалённых строк: ', deleted_share)

Исходное количество строк в датасете:  16956
Количество строк после предобработки данных:  16522

Количество удалённых строк:  434
Доля удалённых строк:  0.0256


**Промежуточные выводы.**

В данных встретилось довольно много явных дубликатов - это неудивительно, так как команда всё собирала вручную. Также в рейтингах встретилась ошибка, а в жанрах - повторяющиеся названия, отличающиеся только регистром. Однако благодаря грамотной предобработке удалось сохранить почти 98% от исходного количества данных. Теперь с ними можно работать. 

<a class="anchor" id="3-bullet"></a>
## 3. Фильтрация данных

Коллеги хотят изучить историю продаж игр в начале XXI века, и их интересует период с 2000 по 2013 год включительно. Нужно отобрать данные по этому показателю и сохранить новый срез данных в отдельном датафрейме.

In [27]:
# Создадим новый датафрейм, в который войдёт только конкретный временной период
games_actual = games_no_duplicates[(games_no_duplicates['year_of_release'] >= 2000)
    & (games_no_duplicates['year_of_release'] <= 2013)]

print('Количество строк в актуальном датафрейме: ', games_actual.shape[0], end='\n\n')
print('Первые строки актуального датафрейма: ', games_actual.head(), sep='\n\n')

Количество строк в актуальном датафрейме:  12856

Первые строки актуального датафрейма: 

                              name platform  year_of_release         genre  \
3906          Frozen: Olaf's Quest       DS             2013      platform   
3394          Frozen: Olaf's Quest      3DS             2013      platform   
2478             Tales of Xillia 2      PS3             2012  role-playing   
8460    .hack//G.U. Vol.1//Rebirth      PS2             2006  role-playing   
7182  .hack//G.U. Vol.2//Reminisce      PS2             2006  role-playing   

      sales_north_america  sales_europe  sales_japan  sales_other  \
3906                 0.21          0.26         0.00         0.04   
3394                 0.27          0.27         0.00         0.05   
2478                 0.20          0.12         0.45         0.07   
8460                 0.00          0.00         0.17         0.00   
7182                 0.11          0.09         0.00         0.03   

      critic_score  user_s

<a class="anchor" id="4-bullet"></a>
## 4. Категоризация данных

<a class="anchor" id="4.1-bullet"></a>
#### Выделяем категории в данных

В данных за выбранный период (с 2000 по 2013 годы включительно) игры нужно разделить на категории:

- **По оценкам пользователей**: `высокая оценка` (от 8 до 10 включительно), `средняя оценка` (от 3 до 8, не включая правую границу интервала) и `низкая оценка` (от 0 до 3, не включая правую границу интервала).
- **По оценкам критиков**: `высокая оценка` (от 80 до 100 включительно), `средняя оценка` (от 30 до 80, не включая правую границу интервала) и `низкая оценка` (от 0 до 30, не включая правую границу интервала).


Начнём с **оценок пользователей**.

In [28]:
# Посмотрим, как выглядят оценки пользователей, всё ли с ними в порядке
print('Начало датафрейма: ', games_actual.loc[:, 'user_score'].sort_values().head(10), sep='\n\n', end='\n\n')
print('Конец датафрейма: ', games_actual.loc[:, 'user_score'].sort_values().tail(10), sep='\n\n', end='\n\n')

# Разобьём данные на категории
games_actual.loc[:, 'user_category'] = pd.cut(games_actual.loc[:, 'user_score'], 
                                              bins=[0, 3, 8, 10], 
                                              labels=["низкая оценка", "средняя оценка", "высокая оценка"], 
                                              right=False)
print('Количество категорий по оценкам пользователей: ', games_actual.loc[:, 'user_category'].value_counts(), sep='\n\n', end='\n\n') 
print('Количество значений, которые не вошли ни в одну категорию: ', games_actual.loc[:, 'user_category'].isna().sum())

Начало датафрейма: 

3906    -1.0
11619   -1.0
12307   -1.0
15130   -1.0
10111   -1.0
10480   -1.0
13730   -1.0
15105   -1.0
16089   -1.0
9661    -1.0
Name: user_score, dtype: float64

Конец датафрейма: 

9685     9.4
714      9.4
1659     9.5
10245    9.5
16784    9.5
16866    9.5
5998     9.5
3454     9.6
9099     9.6
14611    9.7
Name: user_score, dtype: float64

Количество категорий по оценкам пользователей: 

user_category
средняя оценка    4201
высокая оценка    2350
низкая оценка      118
Name: count, dtype: int64

Количество значений, которые не вошли ни в одну категорию:  6187


/var/folders/wx/gd5mkv752w9558dws3m728_00000gn/T/ipykernel_32044/3895194739.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  games_actual.loc[:, 'user_category'] = pd.cut(games_actual.loc[:, 'user_score'],


Перейдём к **оценкам критиков**.

In [29]:
# Посмотрим, как выглядят оценки критиков, всё ли с ними в порядке
print('Начало датафрейма: ', games_actual.loc[:, 'critic_score'].sort_values().head(10), sep='\n\n', end='\n\n')
print('Конец датафрейма: ', games_actual.loc[:, 'critic_score'].sort_values().tail(10), sep='\n\n', end='\n\n')

# Разобьём данные на категории
games_actual.loc[:, 'critic_category'] = pd.cut(games_actual.loc[:, 'critic_score'], 
                                              bins=[0, 30, 80, 100], 
                                              labels=["низкая оценка", "средняя оценка", "высокая оценка"], 
                                              right=False)
print('Количество категорий по оценкам критиков: ', games_actual.loc[:, 'critic_category'].value_counts(), sep='\n\n', end='\n\n') 
print('Количество значений, которые не вошли ни в одну категорию: ', games_actual.loc[:, 'critic_category'].isna().sum())

Начало датафрейма: 

3906    -1
8306    -1
4754    -1
8138    -1
9721    -1
3977    -1
11152   -1
6424    -1
4572    -1
10807   -1
Name: critic_score, dtype: int8

Конец датафрейма: 

49      97
16      97
23      97
1896    97
129     97
519     97
249     97
57      98
227     98
51      98
Name: critic_score, dtype: int8

Количество категорий по оценкам критиков: 

critic_category
средняя оценка    5460
высокая оценка    1704
низкая оценка       56
Name: count, dtype: int64

Количество значений, которые не вошли ни в одну категорию:  5636


/var/folders/wx/gd5mkv752w9558dws3m728_00000gn/T/ipykernel_32044/3909722285.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  games_actual.loc[:, 'critic_category'] = pd.cut(games_actual.loc[:, 'critic_score'],


<a class="anchor" id="4.2-bullet"></a>
#### Составляем топ-7 платформ по количеству игр

Далее выделим топ-7 платформ по количеству игр, выпущенных за весь актуальный период (с 2000 по 2013 годы включительно).

In [30]:
# Посчитаем, сколько игр вышло на каждой платформе
games_on_platforms = games_actual['platform'].value_counts()

# Отсортируем платформы в порядке убывания количества игр
games_on_platforms_sorted = games_on_platforms.sort_values(ascending=False)

print('Топ-7 платформ по количеству игр за 2000-2013 годы: ', games_on_platforms_sorted.head(7), sep='\n\n') 

Топ-7 платформ по количеству игр за 2000-2013 годы: 

platform
PS2     2129
DS      2126
Wii     1286
PSP     1189
X360    1130
PS3     1093
XB       814
Name: count, dtype: int64


<a class="anchor" id="5-bullet"></a>
## Общий вывод из исследования

В этом проекте мы подготовили данные для статьи об эволюции игровой индустрии в начале XXI века. Вот что мы сделали:

**Как предобработали данные**

Сначала мы загрузили датасет и внимательно его изучили. Обнаружили проблемы с типами данных, пропусками и некорректными значениями — например, в столбцах с продажами встречалось 'unknown', а в пользовательских оценках — 'tbd'.

Мы привели названия столбцов к удобному формату `snake_case` и занялись пропусками. Удалили всего 2 строки без названий игр, а пропущенные годы выпуска постарались восстановить — искали ту же игру на других платформах и подставляли оттуда год. То же сделали с жанрами, оценками и рейтингами. То, что восстановить не получилось (в основном оценки и рейтинги), заменили на -1, чтобы не терять данные и иметь возможность работать с числами.

Потом привели типы данных в порядок:

- числовые столбцы сделали `float64`, `int8` или `int16`;
- платформы, жанры и рейтинги перевели в category — так с ними удобнее работать и они занимают меньше памяти.
Нашли и исправили неявные дубликаты: например, жанры были записаны в разном регистре (Action и ACTION) — привели всё к нижнему регистру. Явные дубликаты (237 штук) удалили. В итоге сохранили почти 98% исходных данных — из 16956 строк осталось 16522.

**Какой срез данных получили**

Для анализа оставили только игры, вышедшие с 2000 по 2013 год, — именно этот период интересует команду. Получился датасет games_actual из 12856 записей.

**Что нового добавили**

Разбили игры на категории по оценкам:

По оценкам пользователей: `низкая` (0–3), `средняя` (3–8), `высокая` (8–10) — сохранили в столбце `user_category`.
По оценкам критиков: `низкая` (0–30), `средняя` (30–80), `высокая` (80–100) — сохранили в столбце `critic_category`.
Значения -1 (когда оценки не было) ни в одну категорию не попали — в новых столбцах на их месте пропуски.

Новый датасет выглядит так:

In [31]:
print(games_actual.head())

                              name platform  year_of_release         genre  \
3906          Frozen: Olaf's Quest       DS             2013      platform   
3394          Frozen: Olaf's Quest      3DS             2013      platform   
2478             Tales of Xillia 2      PS3             2012  role-playing   
8460    .hack//G.U. Vol.1//Rebirth      PS2             2006  role-playing   
7182  .hack//G.U. Vol.2//Reminisce      PS2             2006  role-playing   

      sales_north_america  sales_europe  sales_japan  sales_other  \
3906                 0.21          0.26         0.00         0.04   
3394                 0.27          0.27         0.00         0.05   
2478                 0.20          0.12         0.45         0.07   
8460                 0.00          0.00         0.17         0.00   
7182                 0.11          0.09         0.00         0.03   

      critic_score  user_score rating   user_category critic_category  
3906            -1        -1.0   -1.0       

**Что выяснили о платформах**

Посчитали, на каких платформах за 2000–2013 годы вышло больше всего игр. Вот топ-7:

1. `PS2` — 2129 игр
2. `DS` — 2126 игр
3. `Wii` — 1286 игр
4. `PSP` — 1189 игр
5. `X360` — 1130 игр
6. `PS3` — 1093 игры
7. `XB` — 814 игр

Итоговый датасет `games_actual` включает все исходные поля (после предобработки) и два дополнительных категориальных столбца: `user_category` и `critic_category`. Данные готовы для дальнейшего анализа и написания статьи об эволюции игровой индустрии в начале XXI века.